# My examples: automatic finite fields, Ext and Ext0

Version 0.4.5. Edit p and n in section 1a and Phi/Psi in section 2. Run All after changing the field. The symbolic filename is retained; choose mode="symbolic" for formal Frobenius.

## 1. Load the package
Keep the notebook and source files in the extracted package folder. Restart the kernel after installing a new version.

In [1]:
from pathlib import Path
import sys
project_dir = Path.cwd()
if not (project_dir / "fj_sympy").is_dir():
    raise RuntimeError("Start Jupyter in the extracted FJReduction-SymPy folder.")
sys.path.insert(0, str(project_dir))
import sympy as sp
import fj_sympy as fj
from IPython.display import display
from ExampleReports import *
print("FJReduction version:", fj.FJReductionVersion)
assert fj.FJReductionVersion == "0.4.5"
theta, tau, alpha, a, b, c = sp.symbols("theta tau alpha a b c")
def parse_module(rows):
    aliases = {"theta": theta, "tau": tau, "θ": theta, "τ": tau, "FJTwist": fj.FJTwist, "alpha": alpha, "α": alpha}
    return sp.Matrix([[sp.sympify(entry, locals=aliases) for entry in row] for row in rows])


FJReduction version: 0.4.5


## 1a. Choose your field automatically
Set any prime p and positive integer n. Concrete mode constructs F_(p**n)(theta) automatically; no modulus input is needed. For n>1 use alpha for the finite-field generator. Other coefficient symbols such as a and b are transcendental variables, so the field is enlarged to F_(p**n)(theta,a,b,...). Use numerical or alpha-based coefficients to stay in F_(p**n)(theta). After changing the settings, Run All. Finite-field construction is performed inside section 3 before every calculation.

In [2]:
# FIELD SETTINGS: edit only p and n to choose F_(p**n).
p = 3
n = 1
mode = "symbolic"  # "concrete" uses q=p**n; "symbolic" keeps formal Frobenius.


## 2. Your input matrices
Use coefficients on the left of tau, `**` for powers and `*` for multiplication. The source must have an FJ basis, nilpotent constant parts and deg(Psi) smaller than its minimum jump degree. General coefficient symbols are not Frobenius-fixed. Strings such as `"theta + u*tau**3"` introduce new symbols automatically.

In [3]:
# EDIT ONLY THESE TWO MATRICES for your own example.
Phi = [[theta + tau**4, tau], [0, theta + tau**3]]
Psi = [[theta + a*tau, 0], [1 + tau, theta + b*tau]]

Phi=[[theta + theta*tau + theta*tau**4 + tau**3, a*tau**2 + theta*tau**3 + tau + tau**4], [a*tau**3 + theta*tau**2 + tau + tau**4, a*tau**4 + theta + theta*tau**3 + tau]]
Psi=[[a*tau + theta, a*tau + a*tau**2 + a*tau**3, a + a*tau**2 + a*tau**3 + tau], [theta*tau + theta*tau**2, theta + theta*tau, a*tau + theta*tau**2 + 1 + tau**3], [a*tau**2 + tau**3, a*tau**3 + theta*tau**2, theta + theta*tau**3 + tau + tau**2]]


## 3. Compute Ext, the true Ext0 and exact sequence
BEGIN/END comments mark the calculation. Each input change creates fresh parameters. Invalid input clears prior results and reports the actual condition.

In [4]:
# Clear old results before computing the current inputs.
example_result = data = sequence = None
calculation_error = None
try:
    Phi_input, Psi_input = parse_module(Phi), parse_module(Psi)
    if mode == "concrete":
        extra_symbols = sorted(
            (Phi_input.free_symbols | Psi_input.free_symbols) - {theta, tau, alpha},
            key=str)
        parameters = fj.FJParametersPN(p, n, theta=theta, tau=tau,
                                      alpha=alpha, coefficient_symbols=extra_symbols)
        print(f"Field settings: p={p}, n={n}, q={parameters.q}")
        print("Coefficient field:", parameters.field)
        if parameters.field.extension_modulus is not None:
            print("Automatic modulus:", parameters.field.extension_modulus.as_expr())
        if extra_symbols:
            print("Additional coefficient variables:", extra_symbols,
                  "(transcendental; Frobenius acts by q-th powers)")
    elif mode == "symbolic":
        parameters = fj.FJSymbolicParameters(theta=theta, tau=tau)
        print("Formal Frobenius mode: p and n are not specialized.")
    else:
        raise ValueError('mode must be "concrete" or "symbolic"')
    # ===== BEGIN: computation of the true Ext_0 and exact sequence =====
    example_result = FJExampleCompute(Phi_input, Psi_input, parameters)
    data, sequence = example_result["Data"], example_result["Sequence"]
    # ===== END: computation of the true Ext_0 and exact sequence =====
except (fj.FJError, ValueError, TypeError, ZeroDivisionError) as error:
    calculation_error = error
    print("The selected hypotheses do not permit this input:", error)
else:
    print("Dimensions (Ext, Ext0, quotient):", tuple(sequence[k] for k in ("FullDimension", "ExtTau0Dimension", "H")))
    assert sequence["ExactSequenceVerified"]


Formal Frobenius mode: p and n are not specialized.
Dimensions (Ext, Ext0, quotient): (24, 20, 4)


## 4. Descriptive output
Ext is displayed first, followed by the true Ext0 and exact sequence. The report numbers basis representatives and explains matrix shapes, roles and every verification identity. Raw objects remain in `example_result["Sequence"]`.

In [5]:
if example_result is not None:
    display(FJExampleReport(example_result))
else:
    print("No valid result:", calculation_error)

Input and conventions

Phi_t
Input: the action of t on Phi.
Matrix size: 2 x 2
[tau**4*theta + tau**3 + tau*theta + theta, a*tau**2 + tau**4 + tau**3*theta + tau]
[a*tau**3 + tau**4 + tau**2*theta + tau, a*tau**4 + tau**3*theta + tau + theta]

Psi_t
Input: the action of t on Psi.
Matrix size: 3 x 3
[a*tau + theta, a*tau**3 + a*tau**2 + a*tau, a*tau**3 + a*tau**2 + a + tau]
[tau**2*theta + tau*theta, tau*theta + theta, a*tau + tau**3 + tau**2*theta + 1]
[a*tau**2 + tau**3, a*tau**3 + tau**2*theta, tau**3*theta + tau**2 + tau + theta]
d = dim Phi = 2; e = dim Psi = 3. Coordinates are column vectors.
An entry c tau^r is the additive map x |-> c x^(q^r). FJTwist[c,r] denotes the formal coefficient c^(q^r). Use TMult for Ore products, including Frobenius twists.

C: compatible basis
Columns are selected compatible basis vectors. Jump degrees give the normal degree bounds.
Matrix size: 2 x 2
[theta, 1]
[1, a]
sigma | [1, 2]
Jump degrees j_s | [4, 4]
Selected vectors / blocks | [(1, 4, (theta, 1)), (2, 4, (1, a))]

Ext: full normal coordinates
M = e * sum j_s = 24. E(i,s) has a single 1 at (i,s); retain 0 <= r < j_s. The numbering is the row/column order of Pi.
1: E(1,1) tau^0: [[1, 0], [0, 0], [0, 0]]
2: E(1,1) tau^1: [[tau, 0], [0, 0], [0, 0]]
3: E(1,1) tau^2: [[tau**2, 0], [0, 0], [0, 0]]
4: E(1,1) tau^3: [[tau**3, 0], [0, 0], [0, 0]]
5: E(1,2) tau^0: [[0, 1], [0, 0], [0, 0]]
6: E(1,2) tau^1: [[0, tau], [0, 0], [0, 0]]
7: E(1,2) tau^2: [[0, tau**2], [0, 0], [0, 0]]
8: E(1,2) tau^3: [[0, tau**3], [0, 0], [0, 0]]
9: E(2,1) tau^0: [[0, 0], [1, 0], [0, 0]]
10: E(2,1) tau^1: [[0, 0], [tau, 0], [0, 0]]
11: E(2,1) tau^2: [[0, 0], [tau**2, 0], [0, 0]]
12: E(2,1) tau^3: [[0, 0], [tau**3, 0], [0, 0]]
13: E(2,2) tau^0: [[0, 0], [0, 1], [0, 0]]
14: E(2,2) tau^1: [[0, 0], [0, tau], [0, 0]]
15: E(2,2) tau^2: [[0, 0], [0, tau**2], [0, 0]]
16: E(2,2) tau^3: [[0, 0], [0, tau**3], [0, 0]]
17: E(3,1) tau^0: [[0, 0], [0, 0], [1, 0]]
18: E(3,1) tau^1: [[0, 0], [0, 0], [tau, 0]]
19: E(3,1) tau^2: [[0, 0], [0, 0], [tau**2, 0]]
20: E(3,1) tau^3: [[0, 0], [0, 0], [tau**3, 0]]
21: E(3,2) tau^0: [[0, 0], [0, 0], [0, 1]]
22: E(3,2) tau^1: [[0, 0], [0, 0], [0, tau]]
23: E(3,2) tau^2: [[0, 0], [0, 0], [0, tau**2]]
24: E(3,2) tau^3: [[0, 0], [0, 0], [0, tau**3]]

Pi: action of t on Ext
Column j is the image of coordinate j. Powers of tau encode Frobenius powers of that coordinate.
Matrix size: 24 x 24
[theta, a**2*tau**3/(a*theta - 1), a**3*tau**2/(a*theta - 1) + tau**6*(a**2*theta*a^(4) + a*theta)/(a*theta*a^(4)*theta^(4) - a*theta - a^(4)*theta^(4) + 1) + tau**3*(a*a^(1)*theta^(1) - a*theta^(1))/(a*theta*a^(1)*theta^(1) - a*theta - a^(1)*theta^(1) + 1), tau**9*(a**3*a^(4)*a^(8) + a**3 + a**2*theta*a^(4)*a^(8)*theta^(3) + a**2*theta*theta^(3) + a**2*a^(3)*a^(4)*a^(8) + a**2*a^(3) + a**2*a^(8) + a**2*theta^(4) + a*theta*a^(8)*theta^(3) + a*theta*theta^(3)*theta^(4) + a*a^(3)*a^(8) + a*a^(3)*theta^(4))/(a*theta*a^(4)*a^(8)*theta^(4)*theta^(8) - a*theta*a^(4)*theta^(4) - a*theta*a^(8)*theta^(8) + a*theta - a^(4)*a^(8)*theta^(4)*theta^(8) + a^(4)*theta^(4) + a^(8)*theta^(8) - 1) + tau**6*(a**2*theta*a^(1)*a^(4)*theta^(1)*theta^(4) + a**2*theta*a^(1)*a^(5)*theta^(1)*theta^(5) - a**2*theta*a^(1)*theta^(1)*theta^(5) - a**2*theta*a^(1)*theta^(1) - a**2*theta*a^(4)*theta^(4) - a**2*theta*a^(5)*theta^(5) + a**2*theta*theta^(5) + a**2*theta + a*theta*a^(1)*a^(4)*a^(5)*theta^(1)*theta^(4) + a*theta*a^(1)*a^(5)*theta^(1)*theta^(4)*theta^(5) - a*theta*a^(1)*a^(5)*theta^(1) - a*theta*a^(1)*theta^(1)*theta^(4)*theta^(5) - a*theta*a^(4)*a^(5)*theta^(1)*theta^(4) - a*theta*a^(4)*theta^(1)**2*theta^(4) + a*theta*a^(4)*theta^(1)*theta^(4) + a*theta*a^(5)*theta^(1) - a*theta*a^(5)*theta^(4)*theta^(5) + a*theta*theta^(1)**2 - a*theta*theta^(1) + a*theta*theta^(4)*theta^(5))/(a*theta*a^(1)*a^(4)*a^(5)*theta^(1)*theta^(4)*theta^(5) - a*theta*a^(1)*a^(4)*theta^(1)*theta^(4) - a*theta*a^(1)*a^(5)*theta^(1)*theta^(5) + a*theta*a^(1)*theta^(1) - a*theta*a^(4)*a^(5)*theta^(4)*theta^(5) + 

## 5a. Partial basis for your input
When applicable, W and its additive-polynomial inverse verify the same subgroup in different coordinates. Otherwise, the precise failed conditions are explained; the general Ext0 result remains available.

In [ ]:
partial_comparison = None
if example_result is None:
    print("First compute a valid input in section 3.")
else:
    partial_comparison = FJPartialBasisComparison(data, sequence)
    display(FJPartialBasisReport(partial_comparison))


## 5b. J2 counterexample with upper-right 1+tau in Psi
Here Phi=theta I2+J2+tau^2 I2 and Psi=theta I2+J2+J2 tau. The direction count is 1, but h=2 and dim Ext0=6. The candidate count 7 is invalid. The quotient action is nonscalar. For every c, the candidate parameters (0,c,-c^q,0,0,0,c) give the zero class.

In [7]:
# ===== BEGIN: computation of the true Ext_0 and exact sequence =====
j2_theta,j2_tau = sp.symbols("theta tau")
j2_parameters = (fj.FJParametersPN(p, n, theta=j2_theta, tau=j2_tau)
                 if mode == "concrete" else None)
j2Counterexample = FJJ2Example(j2_theta, j2_tau, parameters=j2_parameters)
# ===== END: computation of the true Ext_0 and exact sequence =====
display(j2Counterexample["Output"])
summary = j2Counterexample["Summary"]
assert tuple(summary[k] for k in ("FullDimension", "TrueExt0Dimension", "H", "IndividualZeroCount", "InvalidCountPrediction")) == (8, 6, 2, 1, 7)
assert summary["PartialBasisStatus"] == "Unavailable"
assert j2Counterexample["Result"]["Sequence"]["ExactSequenceVerified"]


Computed quantity,Result
FullDimension,8
TrueExt0Dimension,6
H,2
IndividualZeroCount,1
InvalidCountPrediction,7
PartialBasisStatus,Unavailable
τ2 + θ,1
0,τ2 + θ
θ,τ + 1
0,θ
